# Modeling (LDA) & Social Network Analysis
### Fansoli Ibnu Mustafa | 2411500362
### Dr. Indra, S.Kom., M.T.I.

#### Preprocessing

##### Install and Import Requirements

In [1]:
!pip install gensim nltk scikit-learn emoji pandas Sastrawi networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 13.5 MB/s eta 0:00:00


In [2]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
import re
import numpy as np
import gensim
import nltk
import emoji
import networkx as nx

from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from gensim.corpora import Dictionary, MmCorpus
from gensim.models import LdaModel

from sklearn.cluster import KMeans
from sklearn.manifold import MDS
from sklearn.decomposition import PCA

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [3]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

True

In [4]:
stop_words = set(stopwords.words('indonesian'))

factory = StemmerFactory()
stemmer = factory.create_stemmer()

##### Open Dataset and Preprocessing

In [5]:
df = pd.read_csv('/content/prabowo.csv')
print(f"Data awal: {df.shape[0]} baris")
print("Kolom:", df.columns.tolist())

print("\n",df)

Data awal: 520 baris
Kolom: ['conversation_id_str', 'created_at', 'favorite_count', 'full_text', 'id_str', 'image_url', 'in_reply_to_screen_name', 'lang', 'location', 'quote_count', 'reply_count', 'retweet_count', 'tweet_url', 'user_id_str', 'username']

      conversation_id_str                created_at  favorite_count  \
0    2069943682229317673  2026-06-25T07:21:59.000Z               0   
1    2070044465461690803  2026-06-25T07:20:34.000Z               0   
2    2070044320640737776  2026-06-25T07:20:00.000Z               0   
3    2070043804309008569  2026-06-25T07:17:57.000Z               0   
4    2070014436908421259  2026-06-25T07:15:49.000Z               0   
..                   ...                       ...             ...   
515  2069415866740232393  2026-06-23T13:42:45.000Z              11   
516  2069337391153000549  2026-06-23T13:42:38.000Z               1   
517  2069415231873573346  2026-06-23T13:40:13.000Z               3   
518  2069404914909126681  2026-06-23T13:36:1

In [6]:
text_column = 'full_text'

kamus_alay = {
    "yg": "yang", "dr": "dari", "dri": "dari", "utk": "untuk", "krn": "karena", "karna": "karena","karnaa": "karena", "krna": "karena",

    "gk": "tidak",
    "ga": "tidak",
    "gak": "tidak",
    "nggak": "tidak",
    "enggak": "tidak",
    "kaga": "tidak",
    "tdk": "tidak",

    # Banget
    "bgt": "banget",
    "bgtt": "banget",
    "bangettt": "banget",

    # Sudah
    "udh": "sudah",
    "sdh": "sudah",
    "udhh": "sudah",

    # Lagi
    "lg": "lagi",

    # Tapi
    "tp": "tapi",

    # Jadi
    "jd": "jadi",

    # Saja
    "aja": "saja",
    "aj": "saja",

    # Memang
    "emg": "memang",

    # Kalau
    "kl": "kalau",
    "klo": "kalau",
    "klu": "kalau",

    # Sama
    "sm": "sama",

    # Bisa
    "bs": "bisa",

    # Orang
    "org": "orang",

    # Presiden
    "pres": "presiden",

    # Pemerintah
    "pem": "pemerintah",

    # Indonesia
    "indo": "indonesia",

    # Tolong
    "pls": "tolong",
    "plis": "tolong",

    # Dan lain-lain
    "dll": "dan lain lain"
}

multiword_context = [

    # Tokoh
    "prabowo subianto",
    "presiden prabowo",

    # Pidato
    "pidato prabowo",
    "pidato presiden",

    # Pemerintahan
    "kabinet merah putih",
    "program pemerintah",
    "kebijakan pemerintah",

    # MBG
    "makan bergizi gratis",
    "program makan bergizi",
    "makan gratis",
    "program mbg",
    "mbg 2025",

    # Politik
    "wakil presiden",
    "presiden republik",
    "republik indonesia",

    # Demo
    "demo mahasiswa",
    "aksi mahasiswa",
    "unjuk rasa",

    # Ekonomi
    "harga pangan",
    "harga beras",
    "harga sembako",

    # Pendidikan
    "uang kuliah",
    "biaya kuliah",
    "dana pendidikan",

    # Umum
    "terima kasih",
    "orang indonesia",
    "masyarakat indonesia",
    "rakyat indonesia"
]

In [7]:
def full_preprocessing(text):
  if not isinstance(text, str):
    return "", []

  text = text.lower().strip()
  text = re.sub(r'http\S+|www\S+|https\S+','', text)
  text = re.sub(r'@\w+','', text)
  text = re.sub(r'#\w+','', text)
  text = re.sub(r'[^\w\s]', ' ', text)

  text = emoji.demojize(text, delimiters=("",""))

  # Normalisasi slang
  words = text.split()
  words = [kamus_alay.get(word, word) for word in words]
  text = " ".join(words)

  # Penanganan negasi
  negasi = ["tidak", "ga", "gak", "bukan", "tak", "nggak"]
  tokens = text.split()
  for i in range(len(tokens)-1):
    if tokens[i] in negasi:
      tokens[i] = tokens[i] + "_" + tokens[i+1]
      tokens[i+1] = ""
  text = " ".join([t for t in tokens if t])

  # Repeated characters
  text = re.sub(r'(.)\1{2,}', r'\1\1', text)

  # Context-aware tokenization
  normalized_text = text
  for phrase in multiword_context:
    normalized_text = normalized_text.replace(phrase, phrase.replace(" ", "_"))

  # Custom stopword
  custom_stopwords = {
      "gue", "gua", "gw",
      "banget", "liat", "lihat",
      "ya", "lu", "orang",
      "kalo", "kayak",
      "prabowo", "pidato"
      "dah", "deh"
  }

  # Gabungkan dengan stopword bawaan
  stop_words.update(custom_stopwords)

  # Tokenisasi
  tokens = word_tokenize(normalized_text)
  tokens = [t.replace("_", " ") for t in tokens if len(t) > 1]

  # Hapus stopword
  tokens = [t for t in tokens if t not in stop_words]

  # Stemming
  tokens = [stemmer.stem(token) for token in tokens]

  return normalized_text, tokens

In [8]:
results = df["full_text"].fillna("").apply(full_preprocessing)

df["clean_text"] = results.apply(lambda x: x[0])
df["tokens"] = results.apply(lambda x: x[1])

In [9]:
df[["username","full_text","clean_text","tokens"]].head(10)

,username,full_text,clean_text,tokens
0,Ambisbersama,@tanyarlfes soalny Prabowo tiap pidato cosplay...,soalny prabowo tiap pidato cosplay jadi dementor,"[soalny, pidato, cosplay, dementor]"
1,Tdhziaf,"Gue curiga, di dinasti nya Prabowo ini emang d...",gue curiga di dinasti nya prabowo ini emang di...,"[curiga, dinasti, nya, emang, asli, ken, lengs..."
2,KompasTV,[FULL] IHSG Terkoreksi saat Presiden Prabowo P...,full ihsg terkoreksi saat presiden_prabowo pid...,"[full, ihsg, koreksi, presiden prabowo, pidato..."
3,IRHMPTRA,"Saatnya pulang ke Indonesia, BM pertama kali p...",saatnya pulang ke indonesia bm pertama kali pu...,"[pulang, indonesia, bm, kali, pulang, indonesi..."
4,bbarbieie,"@boxxtoc aku liat pidato nya kemarin, ada adeg...",aku liat pidato nya kemarin ada adegan si prab...,"[pidato, nya, kemarin, adegan, si, manggil, te..."
5,cimandawang,Skrg udah bisa ngeprediksi ihsg bakal turun ka...,skrg udah bisa ngeprediksi ihsg bakal turun ka...,"[skrg, udah, ngeprediksi, ihsg, turun, pidato,..."
6,happyholidayyay,Ya allah malu bgt liat pidato prabowo,ya allah malu banget liat pidato_prabowo,"[allah, malu, pidato prabowo]"
7,ricchiera,"Habis lewat pidato Prabowo kali ini, dari semu...",habis lewat pidato_prabowo kali ini dari semua...,"[habis, pidato prabowo, kali, pidato, edar, pi..."
8,ant00nioo,kayaknya waktu yg tepat beli saham itu waktu p...,kayaknya waktu yang tepat beli saham itu waktu...,"[kayak, beli, saham, pidato, situ, saham, disk..."
9,pacarnyamatty,Ini lama2 gue beneran sakit kepala dengerin Pr...,ini lama2 gue beneran sakit kepala dengerin pr...,"[lama2, beneran, sakit, kepala, dengerin, pidato]"


In [10]:
df.to_csv("hasil-preprocessing_pidato-prabowo.csv", index=False, encoding="utf-8-sig")

print("Berhasil disimpan.")

Berhasil disimpan.


#### Modeling

In [11]:
tweets = df["tokens"].tolist()

print(f"Jumlah dokumen : {len(tweets)}")
print("Contoh token:")
print(tweets[0])

Jumlah dokumen : 520
Contoh token:
['soalny', 'pidato', 'cosplay', 'dementor']


In [12]:
dictionary = Dictionary(tweets)

print(f"Jumlah kata unik : {len(dictionary)}")

Jumlah kata unik : 2008


In [13]:
dictionary.filter_extremes(
    no_below=2,
    no_above=0.9
)

print(f"Jumlah kata setelah filtering : {len(dictionary)}")

Jumlah kata setelah filtering : 639


In [14]:
corpus = [dictionary.doc2bow(tweet) for tweet in tweets]

print(corpus)

[[(0, 1)], [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1)], [(0, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1)], [(22, 3), (23, 1), (24, 1), (25, 1), (26, 2)], [(0, 1), (12, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 2), (32, 1), (33, 1), (34, 1), (35, 1), (36, 1), (37, 1)], [(0, 1), (17, 1), (35, 1), (38, 1), (39, 1)], [(25, 1), (40, 1), (41, 1)], [(0, 3), (23, 2), (25, 1), (42, 1), (43, 1), (44, 1), (45, 1), (46, 1), (47, 1), (48, 1), (49, 1), (50, 1)], [(0, 1), (38, 1), (51, 1), (52, 1), (53, 1), (54, 1), (55, 2), (56, 1), (57, 1)], [(0, 1), (58, 1), (59, 1), (60, 1), (61, 1), (62, 1)], [(0, 1), (40, 1)], [(0, 1), (4, 1), (63, 1), (64, 1), (65, 1), (66, 1)], [(0, 1), (23, 1), (67, 1), (68, 1), (69, 1)], [(0, 1), (22, 1), (70, 1), (71, 1)], [(0, 1), (72, 1), (73, 1), (74, 1), (75, 1), (76, 1)], [(25, 1), (67, 1), (77, 1), (78, 1), (79, 1), (80, 1), (81, 3), (82, 1), (83, 1), (84, 1)], 

In [15]:
num_topics = 5

lda = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    iterations=5000,
)

In [16]:
topics = lda.show_topics(
    formatted=False,
    num_words=10
)

for topic_id, words in topics:
    print(f"\nTopik {topic_id+1}")
    print(", ".join([word for word, prob in words]))


Topik 1
pidato, pidato prabowo, denger, langsung, presiden, nya, nonton, kali, emang, isi

Topik 2
pidato, pidato prabowo, denger, ihsg, maju, kiamat, indonesia, presiden, kali, rupiah

Topik 3
pidato, pidato prabowo, gaji, mbg, anggar, tani, presiden, si, tidak ada, nelayan

Topik 4
pidato, pidato prabowo, indonesia, bikin, kali, maju, ihsg, udah, video, demo

Topik 5
pidato, rakyat, kiamat, maju, pidato prabowo, presiden, muak, denger, bikin, hari


In [17]:
doc_topics = []

for tweet in tweets:
    bow = dictionary.doc2bow(tweet)
    doc_topics.append(
        lda.get_document_topics(bow)
    )

doc_topics[:5]

[[(0, np.float32(0.10345128)),
  (1, np.float32(0.102761485)),
  (2, np.float32(0.10109069)),
  (3, np.float32(0.10254599)),
  (4, np.float32(0.59015054))],
 [(0, np.float32(0.0128712775)),
  (1, np.float32(0.9488618)),
  (2, np.float32(0.012653281)),
  (3, np.float32(0.012931052)),
  (4, np.float32(0.012682665))],
 [(0, np.float32(0.02253025)),
  (1, np.float32(0.023122845)),
  (2, np.float32(0.022329075)),
  (3, np.float32(0.022895453)),
  (4, np.float32(0.90912235))],
 [(0, np.float32(0.18854716)),
  (1, np.float32(0.022953419)),
  (2, np.float32(0.022482129)),
  (3, np.float32(0.74366343)),
  (4, np.float32(0.022353942))],
 [(0, np.float32(0.9453685)),
  (1, np.float32(0.013565399)),
  (2, np.float32(0.013713998)),
  (3, np.float32(0.013690602)),
  (4, np.float32(0.013661608))]]

In [18]:
dominant_topic = []

for topic in doc_topics:
    dominant_topic.append(max(topic, key=lambda x: x[1])[0] + 1)

df["topic"] = dominant_topic

In [19]:
df[["username", "clean_text", "topic"]].head(10)

,username,clean_text,topic
0,Ambisbersama,soalny prabowo tiap pidato cosplay jadi dementor,5
1,Tdhziaf,gue curiga di dinasti nya prabowo ini emang di...,2
2,KompasTV,full ihsg terkoreksi saat presiden_prabowo pid...,5
3,IRHMPTRA,saatnya pulang ke indonesia bm pertama kali pu...,4
4,bbarbieie,aku liat pidato nya kemarin ada adegan si prab...,1
5,cimandawang,skrg udah bisa ngeprediksi ihsg bakal turun ka...,2
6,happyholidayyay,ya allah malu banget liat pidato_prabowo,2
7,ricchiera,habis lewat pidato_prabowo kali ini dari semua...,4
8,ant00nioo,kayaknya waktu yang tepat beli saham itu waktu...,1
9,pacarnyamatty,ini lama2 gue beneran sakit kepala dengerin pr...,2


In [20]:
df.to_csv("hasil-modeling-lda_pidato-prabowo.csv", index=False, encoding="utf-8-sig")

print("Berhasil disimpan.")

Berhasil disimpan.


#### Social Network Analysis

In [27]:
import networkx as nx
import pandas as pd

print("Memproses Graph dan mengimputasi topik untuk akun target...")

# 1. Definisikan edges dari reply_df
reply_df = df[df["in_reply_to_screen_name"].notna()].copy()
edges_df = reply_df[["username", "in_reply_to_screen_name", "topic"]].dropna()
edges_df = edges_df.rename(columns={
    "username": "source",
    "in_reply_to_screen_name": "target",
    "topic": "source_topic"
})

# 2. Cari topik dominan untuk setiap user di dataset utama
known_topics = df.groupby('username')['topic'].agg(lambda x: x.mode()[0]).to_dict()

# 3. Tentukan Topik akun luar berdasarkan MODUS (Paling sering muncul) dari reply
inferred_topics = edges_df.groupby('target')['source_topic'].agg(lambda x: x.mode()[0]).to_dict()

# 4. Buat Directed Graph
G = nx.from_pandas_edgelist(
    edges_df,
    source='source',
    target='target',
    create_using=nx.DiGraph()
)

# 5. Hitung Metrik Centrality
in_degree = dict(G.in_degree())
betweenness = nx.betweenness_centrality(G)

# 6. Setting Warna
color_map = {
    1: {'r': 255, 'g': 99,  'b': 132, 'a': 1.0}, # Merah
    2: {'r': 54,  'g': 162, 'b': 235, 'a': 1.0}, # Biru
    3: {'r': 255, 'g': 206, 'b': 86,  'a': 1.0}, # Kuning
    4: {'r': 75,  'g': 192, 'b': 192, 'a': 1.0}, # Hijau Tosca
    5: {'r': 153, 'g': 102, 'b': 255, 'a': 1.0}  # Ungu
}

# 7. Injeksi Atribut & Visual ke Gephi
for node in G.nodes():
    # Tentukan topik:
    # Prioritas 1: Topik asli dari dataset (jika dia pernah ngetweet)
    # Prioritas 2: Topik inferensi dari reply (jika dia cuma jadi target)
    if node in known_topics:
        topic = int(known_topics[node])
    elif node in inferred_topics:
        topic = int(inferred_topics[node])
    else:
        topic = 1 # Fallback aman

    # Otomatisasi Ukuran Node berdasarkan In-Degree
    # Akun besar yang banyak di-reply akan otomatis jadi bulatan raksasa
    node_size = 10.0 + (in_degree.get(node, 0) * 8.0)

    # Ambil warna
    node_color = color_map.get(topic, {'r': 150, 'g': 150, 'b': 150, 'a': 1.0})

    # Simpan atribut
    G.nodes[node]['Label'] = str(node)
    G.nodes[node]['Topic'] = topic
    G.nodes[node]['In_Degree'] = in_degree.get(node, 0)
    G.nodes[node]['Betweenness'] = betweenness.get(node, 0)

    # Atribut Visual Khusus untuk Gephi (Gak perlu pusing UI gephi lagi)
    G.nodes[node]['viz'] = {'color': node_color, 'size': node_size}

# 8. Export ke format GEXF
file_name = "SNA_Pidato-Prabowo.gexf"
nx.write_gexf(G, file_name)

print(f"Total Nodes : {G.number_of_nodes()}")
print(f"Total Edges : {G.number_of_edges()}")
print(f"Selesai! Download file '{file_name}' dan load ke Gephi.")

Memproses Graph dan mengimputasi topik untuk akun target...
Total Nodes : 248
Total Edges : 165
Selesai! Download file 'SNA_Pidato-Prabowo.gexf' dan load ke Gephi.


Selanjutnya divisualisasikan oleh Gephi.